# Mathematics & Statistics for Machine Learning

A hands-on tutorial for Python programmers who want to build the mathematical
foundation needed to truly **understand** machine learning — not just call
library functions.

## Why does math matter for ML?

| ML Concept | Mathematical Foundation |
|---|---|
| Linear Regression | Linear algebra, calculus, statistics |
| Logistic Regression | Probability, Bayes' theorem |
| Decision Trees | Entropy, information theory |
| Neural Networks | Matrix multiplication, chain rule |
| Clustering | Distance metrics, covariance |
| Evaluation Metrics | Hypothesis testing, distributions |

## What we'll cover

1. **Descriptive Statistics** — summarising data  
2. **Probability Distributions** — modelling uncertainty  
3. **Central Limit Theorem** — why the normal distribution is everywhere  
4. **Hypothesis Testing** — making decisions from data  
5. **Correlation & Covariance** — relationships between features  
6. **Bayes' Theorem** — updating beliefs with evidence  
7. **Linear Algebra Essentials** — the language of ML computation  

## Setup

We use only **NumPy**, **SciPy**, **Matplotlib**, and **Seaborn** — the standard
scientific-Python stack.  All data is synthetic so every result is reproducible.

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
np.random.seed(42)

# Plot style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["figure.dpi"] = 100

print("Setup complete ✓")

---
# 2 · Descriptive Statistics

Descriptive statistics **summarise** a dataset with a few numbers.
They answer: *What does a typical value look like? How spread out are the data?
Is the distribution symmetric?*

These summaries are usually the **first thing** you compute when exploring a
new dataset in an ML project.

## 2.1 Measures of Central Tendency

| Measure | Definition | When to use |
|---|---|---|
| **Mean** | Sum of values / count | Symmetric data, no extreme outliers |
| **Median** | Middle value when sorted | Skewed data or outliers present |
| **Mode** | Most frequent value | Categorical or discrete data |

In [ ]:
# Synthetic dataset: exam scores (slightly left-skewed)
scores = np.concatenate([
    np.random.normal(loc=72, scale=10, size=200),
    np.random.normal(loc=45, scale=5, size=30),   # low-performing group
])
scores = np.clip(scores, 0, 100)

mean_score = np.mean(scores)
median_score = np.median(scores)
# Mode via scipy
mode_result = stats.mode(scores.round(0), keepdims=False)
mode_score = mode_result.mode

print(f"Mean   : {mean_score:.2f}")
print(f"Median : {median_score:.2f}")
print(f"Mode   : {mode_score:.0f}")

> **ML Connection:** When the mean and median diverge, the distribution is
> skewed. Many ML algorithms (e.g. linear regression) assume roughly symmetric
> residuals, so knowing this early helps you decide whether to transform
> features (e.g. log transform).

## 2.2 Measures of Spread

Spread tells us **how variable** the data are. Two datasets can share the same
mean but look completely different if their spreads differ.

In [ ]:
variance = np.var(scores, ddof=1)       # sample variance (Bessel's correction)
std_dev = np.std(scores, ddof=1)        # sample standard deviation
iqr = np.percentile(scores, 75) - np.percentile(scores, 25)
data_range = np.ptp(scores)             # max - min

print(f"Variance          : {variance:.2f}")
print(f"Std Deviation     : {std_dev:.2f}")
print(f"IQR (Q3 − Q1)    : {iqr:.2f}")
print(f"Range (max − min) : {data_range:.2f}")

## 2.3 Shape: Skewness & Kurtosis

* **Skewness** measures asymmetry (0 = symmetric, negative = left tail, positive = right tail).  
* **Kurtosis** measures tail heaviness relative to a normal distribution.
  *Excess* kurtosis = 0 for a normal distribution.

In [ ]:
skew = stats.skew(scores)
kurt = stats.kurtosis(scores)   # excess kurtosis by default

print(f"Skewness         : {skew:.3f}")
print(f"Excess Kurtosis  : {kurt:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(scores, bins=25, edgecolor="white", color="steelblue", alpha=0.8)
axes[0].axvline(mean_score, color="red", ls="--", label=f"Mean {mean_score:.1f}")
axes[0].axvline(median_score, color="orange", ls="--", label=f"Median {median_score:.1f}")
axes[0].set_title("Distribution of Exam Scores")
axes[0].set_xlabel("Score")
axes[0].legend()

# Box plot
axes[1].boxplot(scores, vert=False, widths=0.6,
                patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.6))
axes[1].set_title("Box Plot — Outlier Detection")
axes[1].set_xlabel("Score")

plt.tight_layout()
plt.show()

> **Key take-away:** Descriptive statistics are your *first line of defence*
> against dirty data. Always compute them before training a model.

---
# 3 · Probability Distributions

A **probability distribution** describes how likely each outcome is.

* **PMF** (Probability Mass Function) — for *discrete* distributions (countable outcomes).  
* **PDF** (Probability Density Function) — for *continuous* distributions (any value in an interval).  
* **CDF** (Cumulative Distribution Function) — P(X ≤ x); works for both.

## 3.1 Normal (Gaussian) Distribution

The most important distribution in ML. It arises naturally (Central Limit
Theorem) and underpins many algorithms.

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

In [ ]:
x = np.linspace(-4, 4, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# PDF for different parameters
for mu, sigma in [(0, 1), (0, 2), (2, 1)]:
    axes[0].plot(x, stats.norm.pdf(x, mu, sigma),
                 label=f"μ={mu}, σ={sigma}")
axes[0].set_title("Normal PDF")
axes[0].set_xlabel("x")
axes[0].set_ylabel("Density")
axes[0].legend()

# CDF
axes[1].plot(x, stats.norm.cdf(x, 0, 1), color="steelblue", lw=2)
axes[1].axhline(0.5, color="gray", ls=":", alpha=0.5)
axes[1].axvline(0, color="gray", ls=":", alpha=0.5)
axes[1].set_title("Standard Normal CDF")
axes[1].set_xlabel("x")
axes[1].set_ylabel("P(X ≤ x)")

plt.tight_layout()
plt.show()

## 3.2 Uniform Distribution

Every outcome in [a, b] is equally likely.
Useful as a **baseline / null model** — "no information" about which values are
more probable.

In [ ]:
a, b = 2, 8
x_u = np.linspace(0, 10, 300)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(x_u, stats.uniform.pdf(x_u, loc=a, scale=b - a),
        color="darkorange", lw=2, label="PDF")
ax.fill_between(x_u, stats.uniform.pdf(x_u, loc=a, scale=b - a),
                alpha=0.2, color="darkorange")
ax.set_title(f"Uniform Distribution  U({a}, {b})")
ax.set_xlabel("x")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

## 3.3 Binomial Distribution (Discrete)

Models the **number of successes** in *n* independent Bernoulli trials, each
with success probability *p*.

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

In [ ]:
n_trials, p_success = 20, 0.3
k = np.arange(0, n_trials + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# PMF
axes[0].bar(k, stats.binom.pmf(k, n_trials, p_success),
            color="mediumseagreen", edgecolor="white")
axes[0].set_title(f"Binomial PMF  (n={n_trials}, p={p_success})")
axes[0].set_xlabel("k (successes)")
axes[0].set_ylabel("P(X = k)")

# CDF
axes[1].step(k, stats.binom.cdf(k, n_trials, p_success),
             where="mid", color="mediumseagreen", lw=2)
axes[1].set_title(f"Binomial CDF  (n={n_trials}, p={p_success})")
axes[1].set_xlabel("k")
axes[1].set_ylabel("P(X ≤ k)")

plt.tight_layout()
plt.show()

## 3.4 Poisson Distribution (Discrete)

Models the **count of events** in a fixed interval when events occur
independently at a constant average rate λ.

$$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}$$

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for lam in [1, 4, 10]:
    k_vals = np.arange(0, 25)
    ax.bar(k_vals + lam * 0.05, stats.poisson.pmf(k_vals, lam),
           width=0.8, alpha=0.6, label=f"λ = {lam}")

ax.set_title("Poisson PMF for Different λ")
ax.set_xlabel("k (event count)")
ax.set_ylabel("P(X = k)")
ax.legend()
plt.tight_layout()
plt.show()

> **ML Connection:** Poisson regression is used when the target variable is a
> *count* (e.g., number of website clicks, insurance claims). Understanding
> the distribution helps you pick the right loss function.

---
# 4 · Central Limit Theorem (CLT)

> No matter the shape of the original population, the distribution of
> **sample means** approaches a normal distribution as the sample size grows.

This is arguably the *most important theorem in statistics* — it justifies
using the normal distribution in confidence intervals, hypothesis tests, and
many ML methods, even when the underlying data are not normal.

In [ ]:
# Population: highly skewed exponential
population = np.random.exponential(scale=2.0, size=100_000)

sample_sizes = [2, 5, 30, 100]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, n in zip(axes.ravel(), sample_sizes):
    means = [np.mean(np.random.choice(population, size=n)) for _ in range(2000)]
    ax.hist(means, bins=40, density=True, color="steelblue",
            edgecolor="white", alpha=0.7)
    # Overlay theoretical normal
    mu_hat, sigma_hat = np.mean(means), np.std(means)
    x_fit = np.linspace(min(means), max(means), 200)
    ax.plot(x_fit, stats.norm.pdf(x_fit, mu_hat, sigma_hat),
            "r-", lw=2, label="Normal fit")
    ax.set_title(f"Sample size n = {n}")
    ax.legend(fontsize=9)

fig.suptitle("Central Limit Theorem — Exponential Population", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Observations:**
- With n = 2 the sampling distribution is still skewed.
- By n = 30 it already looks remarkably normal.
- At n = 100 it is almost perfectly Gaussian.

> **ML Connection:** Stochastic gradient descent computes *mini-batch* means
> of gradients. The CLT explains why batch sizes ≥ 30 tend to produce stable,
> approximately normal gradient estimates.

---
# 5 · Hypothesis Testing

Hypothesis testing lets us make **data-driven decisions** under uncertainty.

| Term | Meaning |
|---|---|
| **H₀** (null hypothesis) | Default assumption (e.g., "no difference") |
| **H₁** (alternative) | What we want to show |
| **p-value** | Probability of seeing data this extreme *if H₀ is true* |
| **α (significance level)** | Threshold (commonly 0.05); reject H₀ if p < α |
| **Type I error** | False positive — rejecting H₀ when it's true |
| **Type II error** | False negative — failing to reject H₀ when H₁ is true |

## 5.1 One-sample & Two-sample t-tests

The **t-test** checks whether a sample mean differs significantly from a
hypothesised value (one-sample) or whether two sample means differ from each
other (two-sample).

In [ ]:
# One-sample t-test: is the mean score = 70?
t_stat, p_val = stats.ttest_1samp(scores, popmean=70)
print("=== One-Sample t-Test (H₀: μ = 70) ===")
print(f"  t-statistic : {t_stat:.4f}")
print(f"  p-value     : {p_val:.4f}")
print(f"  Reject H₀?  : {'Yes' if p_val < 0.05 else 'No'} (α = 0.05)")

In [ ]:
# Two-sample t-test: two classes with different teaching methods
np.random.seed(42)
class_a = np.random.normal(loc=74, scale=8, size=50)
class_b = np.random.normal(loc=70, scale=9, size=50)

t2, p2 = stats.ttest_ind(class_a, class_b, equal_var=False)  # Welch's t-test
print("=== Two-Sample t-Test (H₀: μ_A = μ_B) ===")
print(f"  t-statistic : {t2:.4f}")
print(f"  p-value     : {p2:.4f}")
print(f"  Reject H₀?  : {'Yes' if p2 < 0.05 else 'No'} (α = 0.05)")

## 5.2 Chi-Square Test of Independence

Used for **categorical** data. It tests whether two categorical variables are
independent.

In [ ]:
# Contingency table: study method vs pass/fail
# Rows: method (A, B, C)  |  Cols: fail, pass
observed = np.array([
    [15, 85],   # Method A
    [25, 75],   # Method B
    [30, 70],   # Method C
])

chi2, p_chi, dof, expected = stats.chi2_contingency(observed)
print("=== Chi-Square Test of Independence ===")
print(f"  χ² statistic : {chi2:.4f}")
print(f"  p-value      : {p_chi:.4f}")
print(f"  Degrees of freedom : {dof}")
print(f"  Reject H₀ (independence)? : {'Yes' if p_chi < 0.05 else 'No'}")
print(f"\nExpected frequencies:\n{expected.round(1)}")

## 5.3 Visualising the p-value

A p-value is the area under the null distribution *beyond* the observed test
statistic.

In [ ]:
x_t = np.linspace(-5, 5, 400)
df_t = len(scores) - 1  # degrees of freedom from one-sample test
pdf_t = stats.t.pdf(x_t, df=df_t)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x_t, pdf_t, "k-", lw=2)
ax.fill_between(x_t, pdf_t, where=(x_t <= -abs(t_stat)) | (x_t >= abs(t_stat)),
                color="red", alpha=0.35, label=f"p-value = {p_val:.4f}")
ax.axvline(t_stat, color="red", ls="--", lw=1.5, label=f"t = {t_stat:.2f}")
ax.axvline(-t_stat, color="red", ls="--", lw=1.5)
ax.set_title("Two-Tailed t-Test — Null Distribution")
ax.set_xlabel("t")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

> **ML Connection:** In feature selection, statistical tests (e.g., chi-square,
> ANOVA F-test) help rank features by their association with the target. The
> scikit-learn `SelectKBest` transformer uses these tests directly.

---
# 6 · Correlation & Covariance

**Covariance** measures how two variables move together (units depend on the
variables).  
**Correlation** normalises covariance to [−1, +1], making it unit-free and
easier to interpret.

| Correlation | Interpretation |
|---|---|
| +1 | Perfect positive linear relationship |
| 0 | No linear relationship |
| −1 | Perfect negative linear relationship |

## 6.1 Pearson vs Spearman Correlation

In [ ]:
np.random.seed(42)
x = np.random.uniform(0, 10, 100)
y_linear = 2 * x + np.random.normal(0, 2, 100)          # linear
y_monotone = np.exp(0.3 * x) + np.random.normal(0, 1, 100)  # monotone but non-linear

pearson_lin, _ = stats.pearsonr(x, y_linear)
spearman_lin, _ = stats.spearmanr(x, y_linear)
pearson_mon, _ = stats.pearsonr(x, y_monotone)
spearman_mon, _ = stats.spearmanr(x, y_monotone)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(x, y_linear, alpha=0.6, s=25, color="steelblue")
axes[0].set_title(f"Linear: Pearson={pearson_lin:.2f}, Spearman={spearman_lin:.2f}")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")

axes[1].scatter(x, y_monotone, alpha=0.6, s=25, color="darkorange")
axes[1].set_title(f"Monotone: Pearson={pearson_mon:.2f}, Spearman={spearman_mon:.2f}")
axes[1].set_xlabel("x"); axes[1].set_ylabel("y")

plt.tight_layout()
plt.show()

> **Take-away:** Spearman captures *monotonic* relationships that Pearson
> misses when the relationship is non-linear but consistently increasing (or
> decreasing).

## 6.2 Correlation Matrix Heatmap

When you have many features, a correlation matrix instantly reveals which
pairs are strongly related — useful for detecting **multicollinearity** before
training a linear model.

In [ ]:
np.random.seed(42)
n = 200
feature_names = ["Height", "Weight", "Age", "Income", "Spending"]
data = np.column_stack([
    np.random.normal(170, 10, n),                          # Height
    np.random.normal(170, 10, n) * 0.5 + np.random.normal(0, 5, n),  # Weight ≈ corr with Height
    np.random.uniform(20, 65, n),                          # Age
    np.random.exponential(50000, n),                       # Income
    np.random.exponential(50000, n) * 0.6 + np.random.normal(0, 5000, n),  # Spending ≈ corr with Income
])

corr_matrix = np.corrcoef(data, rowvar=False)

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f",
            xticklabels=feature_names, yticklabels=feature_names,
            cmap="coolwarm", center=0, square=True, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## 6.3 Anscombe's Quartet — Why You Must Visualise

Four tiny datasets share **identical** summary statistics (mean, variance,
correlation, regression line) yet look completely different.

**Lesson:** Never trust summary statistics alone — always plot your data.

In [ ]:
# Anscombe's quartet (classic values)
anscombe_x = [
    [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
]
anscombe_y = [
    [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68],
    [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74],
    [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73],
    [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89],
]

fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=False, sharey=False)
for idx, ax in enumerate(axes.ravel()):
    xi, yi = np.array(anscombe_x[idx]), np.array(anscombe_y[idx])
    ax.scatter(xi, yi, color="steelblue", s=50)
    m, b = np.polyfit(xi, yi, 1)
    ax.plot(xi, m * xi + b, "r--", lw=1.5)
    r, _ = stats.pearsonr(xi, yi)
    ax.set_title(f"Dataset {idx+1}  (r = {r:.2f})")
    ax.set_xlabel("x"); ax.set_ylabel("y")

fig.suptitle("Anscombe's Quartet — Same Stats, Different Data", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

> **ML Connection:** This is why exploratory data analysis (EDA) with plots is
> an essential step before feature engineering. Correlation coefficients alone
> can mislead you.

---
# 7 · Bayes' Theorem

Bayes' theorem tells us how to **update** the probability of a hypothesis
when new evidence arrives:

$$P(A \mid B) = \frac{P(B \mid A)\, P(A)}{P(B)}$$

| Term | Name | Meaning |
|---|---|---|
| P(A\|B) | **Posterior** | Updated belief after seeing evidence B |
| P(B\|A) | **Likelihood** | How likely is the evidence if A is true? |
| P(A) | **Prior** | Initial belief before evidence |
| P(B) | **Evidence** | Total probability of observing B |

## 7.1 Medical Test Example

A disease affects **1 in 1 000** people. A test is 99 % sensitive (true
positive rate) and 95 % specific (true negative rate).

If you test **positive**, what is the actual probability you have the disease?

In [ ]:
prevalence = 0.001        # P(Disease)
sensitivity = 0.99        # P(Positive | Disease)
specificity = 0.95        # P(Negative | No Disease)
false_positive_rate = 1 - specificity

# P(Positive) via law of total probability
p_positive = sensitivity * prevalence + false_positive_rate * (1 - prevalence)

# Bayes' theorem
p_disease_given_pos = (sensitivity * prevalence) / p_positive

print("=== Medical Test — Bayes' Theorem ===")
print(f"  Prevalence          : {prevalence:.4f}")
print(f"  Sensitivity (TPR)   : {sensitivity:.2f}")
print(f"  Specificity (TNR)   : {specificity:.2f}")
print(f"  P(Positive)         : {p_positive:.5f}")
print(f"  P(Disease | Pos)    : {p_disease_given_pos:.4f}  ({p_disease_given_pos*100:.2f} %)")
print()
print("→ Even with a 99 % sensitive test, a positive result means only")
print(f"  ~{p_disease_given_pos*100:.1f} % chance of actually having the disease!")

## 7.2 Why is the posterior so low?

The **base rate** (prevalence) is tiny. The 5 % false-positive rate generates
far more false alarms than the disease generates true positives. This is
called the **base-rate fallacy**.

Let's visualise with a natural-frequency tree:

In [ ]:
N = 100_000
has_disease = int(N * prevalence)
no_disease = N - has_disease

true_pos = int(has_disease * sensitivity)
false_neg = has_disease - true_pos
false_pos = int(no_disease * false_positive_rate)
true_neg = no_disease - false_pos

print(f"Population           : {N:>7,}")
print(f"  Has disease        : {has_disease:>7,}")
print(f"    Test positive TP : {true_pos:>7,}")
print(f"    Test negative FN : {false_neg:>7,}")
print(f"  No disease         : {no_disease:>7,}")
print(f"    Test positive FP : {false_pos:>7,}")
print(f"    Test negative TN : {true_neg:>7,}")
print(f"\nOf all positives ({true_pos + false_pos:,}), "
      f"only {true_pos} are true positives.")

## 7.3 Connection to Naive Bayes Classifier

The **Naive Bayes** classifier applies Bayes' theorem to classify data points.
It assumes features are conditionally independent given the class:

$$P(C \mid x_1, x_2, \dots, x_n) \propto P(C) \prod_{i=1}^{n} P(x_i \mid C)$$

Despite the "naive" independence assumption, it works surprisingly well for
text classification, spam filtering, and sentiment analysis.

In [ ]:
# Mini-demo: spam detection intuition with Bayes
# P(spam) = 0.3, P(ham) = 0.7
# Word "free": P("free" | spam) = 0.8, P("free" | ham) = 0.1
p_spam = 0.3
p_ham = 0.7
p_free_spam = 0.8
p_free_ham = 0.1

p_free = p_free_spam * p_spam + p_free_ham * p_ham
p_spam_given_free = (p_free_spam * p_spam) / p_free

print("=== Naive Bayes Spam Example ===")
print(f'  P(spam | "free") = {p_spam_given_free:.3f}  ({p_spam_given_free*100:.1f} %)')
print(f'  P(ham  | "free") = {1 - p_spam_given_free:.3f}  ({(1-p_spam_given_free)*100:.1f} %)')
print()
print('→ Seeing the word "free" shifts our belief from 30 % to',
      f'{p_spam_given_free*100:.1f} % that the email is spam.')

---
# 8 · Linear Algebra Essentials

Linear algebra is the **language of ML computation**. Data are stored as
matrices, transformations are matrix multiplications, and optimisation
involves gradients (vectors).

We'll cover just what you need to get started.

## 8.1 Vectors and Basic Operations

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("Vectors")
print(f"  a = {a}")
print(f"  b = {b}")
print(f"\nElement-wise addition   : a + b = {a + b}")
print(f"Scalar multiplication   : 3·a   = {3 * a}")
print(f"Dot product             : a · b = {np.dot(a, b)}")
print(f"Magnitude (L2 norm)     : ||a|| = {np.linalg.norm(a):.4f}")

## 8.2 Matrices and Multiplication

Matrix multiplication is the **workhorse** of neural networks. Each layer
computes **Y = XW + b** — a matrix multiply plus a bias vector.

In [ ]:
X = np.array([[1, 2],
              [3, 4],
              [5, 6]])       # 3×2

W = np.array([[0.5, -1, 0.2],
              [0.3,  0.8, -0.5]])  # 2×3

Y = X @ W                    # 3×3

print(f"X (3×2):\n{X}\n")
print(f"W (2×3):\n{W}\n")
print(f"Y = X @ W  (3×3):\n{Y}")

## 8.3 Determinant, Inverse, and Rank

In [ ]:
M = np.array([[2, 1],
              [5, 3]])

det_M = np.linalg.det(M)
inv_M = np.linalg.inv(M)
rank_M = np.linalg.matrix_rank(M)

print(f"M:\n{M}\n")
print(f"Determinant : {det_M:.4f}")
print(f"Rank        : {rank_M}")
print(f"Inverse:\n{inv_M}")
print(f"\nVerification  M @ M⁻¹ =\n{(M @ inv_M).round(6)}")

## 8.4 Eigenvalues and Eigenvectors

An **eigenvector** of a matrix *A* is a direction that is only *scaled*
(not rotated) by the transformation:

$$A\mathbf{v} = \lambda \mathbf{v}$$

where λ is the **eigenvalue** (the scaling factor).

In ML, eigenvalues appear in:
- **PCA** — principal components are the eigenvectors of the covariance matrix.
- **Spectral clustering** — uses eigenvectors of the graph Laplacian.
- **PageRank** — the ranking vector is the dominant eigenvector of the web graph.

In [ ]:
# Symmetric positive-definite matrix (like a covariance matrix)
C = np.array([[3.0, 1.5],
              [1.5, 2.0]])

eigenvalues, eigenvectors = np.linalg.eigh(C)

print(f"Covariance matrix C:\n{C}\n")
print(f"Eigenvalues  : {eigenvalues}")
print(f"Eigenvectors (columns):\n{eigenvectors}\n")

# Visualise
fig, ax = plt.subplots(figsize=(6, 6))
np.random.seed(42)
pts = np.random.multivariate_normal([0, 0], C, 300)
ax.scatter(pts[:, 0], pts[:, 1], alpha=0.3, s=15, color="steelblue")

origin = np.array([0, 0])
for i, (val, vec) in enumerate(zip(eigenvalues, eigenvectors.T)):
    ax.annotate("", xy=vec * np.sqrt(val) * 2, xytext=origin,
                arrowprops=dict(arrowstyle="->", lw=2.5,
                                color="red" if i == 1 else "darkorange"))
    ax.text(*(vec * np.sqrt(val) * 2.2),
            f"λ={val:.2f}", fontsize=11, fontweight="bold",
            color="red" if i == 1 else "darkorange")

ax.set_title("Eigenvectors of a 2D Covariance Matrix (PCA directions)")
ax.set_xlabel("x₁"); ax.set_ylabel("x₂")
ax.set_aspect("equal")
ax.set_xlim(-6, 6); ax.set_ylim(-6, 6)
plt.tight_layout()
plt.show()

> **ML Connection:** PCA projects data onto the eigenvectors with the
> **largest** eigenvalues, keeping the directions of maximum variance and
> discarding noise. This is the most common dimensionality-reduction technique.

---
# 9 · Summary & Connections to ML

| Topic | Key Idea | ML Application |
|---|---|---|
| **Descriptive Stats** | Summarise centre, spread, shape | EDA, feature scaling, outlier detection |
| **Distributions** | Model randomness with known forms | Choosing loss functions, generative models |
| **CLT** | Sample means → Normal | Confidence in mini-batch gradients |
| **Hypothesis Testing** | Decide with controlled error rates | Feature selection, A/B testing |
| **Correlation** | Measure linear/monotonic association | Feature selection, multicollinearity |
| **Bayes' Theorem** | Update beliefs with evidence | Naive Bayes, Bayesian optimisation |
| **Linear Algebra** | Vectors, matrices, eigenvalues | Every model: dot products, PCA, neural nets |

## Recommended Next Steps

1. **Calculus for ML** — gradients, partial derivatives, the chain rule  
2. **Optimisation** — gradient descent, learning rates, convergence  
3. **Information Theory** — entropy, cross-entropy, KL divergence  
4. **Hands-on ML** — apply these concepts in scikit-learn projects  

> *"The goal is not to memorise formulas, but to develop intuition for when
> and why each mathematical tool is the right choice."*

---
*Notebook generated programmatically with `nbformat`. All data is synthetic
(`np.random.seed(42)`).*